In [ ]:
# bulkRNA-seq pySCENIC Analysis
# 
# This notebook performs pySCENIC (Single-Cell rEgulatory Network Inference and Clustering) analysis
# on bulk RNA-seq data to identify gene regulatory networks and transcription factors
# 
# pySCENIC workflow consists of three main steps:
# 1. GRN inference using GRNBoost2 algorithm
# 2. Regulon prediction (cisTarget motif analysis) 
# 3. Cellular enrichment (AUCell scoring)

# Standard data manipulation libraries
import os
import glob
import pickle
import pandas as pd
import numpy as np
import requests

# Progress tracking for long-running computations
from dask.diagnostics import ProgressBar

# Gene regulatory network inference libraries
from arboreto.utils import load_tf_names  # Load transcription factor gene names
from arboreto.algo import grnboost2       # GRN inference algorithm

# pySCENIC core libraries
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase  # Motif ranking databases
from pyscenic.utils import modules_from_adjacencies, load_motifs      # Network module processing
from pyscenic.prune import prune2df, df2regulons                     # Motif enrichment and regulon creation
from pyscenic.aucell import aucell                                   # Activity scoring

# Visualization and utilities
import seaborn as sns
import datetime
import matplotlib.pyplot as plt

In [ ]:
# Date configuration for output file naming
# Using current date for timestamping output files, but overridden with specific date for reproducibility
current_date = datetime.datetime.now().strftime("%Y%m%d")  # Generate current date string in YYYYMMDD format

In [ ]:
# Analysis configuration parameters
# Define the target disease and control group settings

# Target disease abbreviation (RP = Recurrent Pericarditis)
TARGET_DISEASE = "RP"

# Whether to include Normal Control (NC) samples in the analysis
# Including controls allows for comparative analysis between disease and healthy states
INCLUDE_NC = True

# String suffix for file naming - will be 'NC_' if controls are included, empty string if not
# This ensures output files are properly labeled to indicate whether controls were included
NC_STR = 'NC_' if INCLUDE_NC else ''

In [ ]:
# File path configuration
# Configure all input and output paths for the pySCENIC analysis pipeline

# Base directory structure - modify these paths according to your data organization
BASE_PATH = '/the-base-data-path/'                                    # Root directory for all data
MATRIX_PATH = BASE_PATH + 'the-relative-path-to-input-matrix/'        # RNA-seq count matrices location
OUTPUT_PATH = BASE_PATH + 'the-relative-path-to-output/'              # Analysis results output directory

# pySCENIC reference data paths
DATABASE_PATH = BASE_PATH + 'the-relative-path-to-databases/'         # Motif ranking databases
RESOURCES_PATH = BASE_PATH + 'the-relative-path-to-pyScenic-data-resources/'  # TF lists and annotations

# pySCENIC reference files (these are standard files downloaded from pySCENIC resources)
DATABASES_GLOB = os.path.join(DATABASE_PATH, 'hg38_*_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather')  # Motif ranking databases (multiple files)
MOTIF_ANNOTATIONS_FNAME = os.path.join(RESOURCES_PATH, 'motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl')  # Motif-to-TF annotations
MM_TFS_FNAME = os.path.join(RESOURCES_PATH, 'hs_hgnc_tfs.txt')  # Human transcription factor gene list

# File naming logic explanation:
# If INCLUDE_NC is True, REGULONS_FNAME and MOTIFS_FNAME file names will contain '_NC_', otherwise not
# If INCLUDE_NC is True, bulkRNA_df will include NC control group data for comparative analysis

# Output file paths - these will store intermediate and final results
# All files include disease type, control group status, and date for proper organization
REGULONS_FNAME = os.path.join(OUTPUT_PATH, 'regulons_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.p')        # Final regulons (pickle format)
MOTIFS_FNAME = os.path.join(OUTPUT_PATH, 'motifs_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.csv')          # Motif enrichment results
ADJACENCIES_FNAME = os.path.join(OUTPUT_PATH, 'adjacencies_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.tsv')  # GRN adjacencies matrix
MODULES_FNAME = os.path.join(OUTPUT_PATH, 'modules_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.p')          # Co-expression modules (pickle format)

# Input data organization - RNA-seq data is organized by batch folders
# Each batch contains gene count files for different samples
batch_paths = {}
batch_paths['batch1'] = MATRIX_PATH + 'Batch1/GeneCounts/'  # First batch of samples
batch_paths['batch2'] = MATRIX_PATH + 'Batch2/GeneCounts/'  # Second batch of samples  
batch_paths['batch3'] = MATRIX_PATH + 'Batch3/GeneCounts/'  # Third batch of samples
batch_paths['batch4'] = MATRIX_PATH + 'Batch4/GeneCounts/'  # Fourth batch of samples
batch_paths['batch5'] = MATRIX_PATH + 'Batch5/GeneCounts/'  # Fifth batch of samples

# Load clinical metadata to identify which samples belong to target disease vs controls
# This file contains patient ID to disease mappings
clinical_data = pd.read_csv('/Users/weiyin/Workspaces/Services/bioinfo/data/464/processed/patients_diseases/immunaid_patients_diseases_20240822.tsv', sep='\t', dtype=str)

# Extract patient IDs for target disease samples
# Filter clinical data to get only patients with the target disease (RP)
patient_ids_TARGET = clinical_data[clinical_data['disease_brief'].isin([TARGET_DISEASE])]['patient_id']

# Extract patient IDs for normal control samples  
# Filter clinical data to get only normal control (NC) patients
patient_ids_NC = clinical_data[clinical_data['disease_brief'].isin(['NC'])]['patient_id']

# Data loading strategy explanation:
# Iterate through each folder in batch_paths, if the first 5 characters of filename are in patient_ids_RP or patient_ids_NC, read the file as DataFrame
# and merge it into one DataFrame: bulkRNA_df
# Each read DataFrame needs to be added as a column to bulkRNA_df, column name is the first part after splitting filename by '_'

# Initialize empty DataFrame to store all gene expression data
bulkRNA_df = pd.DataFrame()

# Data filtering option - whether to accept M3 (technical replicate B) data
# Set to False to use only primary samples (technical replicate A)
should_accept_M3_data = False

In [ ]:
# Data Loading: Load bulk RNA-seq expression data from multiple batches
# 
# This section reads gene expression count files from different batch folders
# and combines them into a single expression matrix for analysis

for batch_name, batch_path in batch_paths.items():
    print(f"Processing {batch_name} from {batch_path}")
    
    # Iterate through all files in the current batch directory
    for file_name in os.listdir(batch_path):
        # Skip hidden files (starting with '.')
        if file_name.startswith('.'):
            continue
            
        # Extract patient ID from filename (first part before underscore)
        patient_id = file_name.split('_')[0]
        
        # Check if this patient belongs to our target disease or control groups
        # Only process files for patients in our study cohorts
        if patient_id[:5] in patient_ids_TARGET.values or (INCLUDE_NC and patient_id[:5] in patient_ids_NC.values):
            
            # Technical replicate filtering: 
            # Files with 6th character 'B' are technical replicates (M3 data)
            # Skip these unless explicitly enabled to avoid duplicate samples
            if file_name[5] == 'B' and not should_accept_M3_data:
                continue

            # Read gene expression count file
            file_path = batch_path + file_name
            # File format: tab-separated, first column is Ensembl gene IDs, second column is counts
            # Original data has no column names, so we manually specify them
            df = pd.read_csv(file_path, sep='\t', index_col=0, header=None, names=['ensembl_id', patient_id])
            
            # Combine this sample's data with the main expression matrix
            # Each sample becomes a column in the final matrix
            bulkRNA_df = pd.concat([bulkRNA_df, df], axis=1)

print(f"Loaded expression data for {bulkRNA_df.shape[1]} samples and {bulkRNA_df.shape[0]} genes")

In [ ]:
# Data Export: Save raw expression matrix with Ensembl IDs
# 
# Save the combined expression matrix before gene ID conversion
# This preserves the original Ensembl gene identifiers for reference
output_file = OUTPUT_PATH + 'bulkRNA_gene_count_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.tsv'
bulkRNA_df.to_csv(output_file, sep='\t', encoding='utf-8')
print(f"Saved raw expression matrix to: {output_file}")

In [ ]:
# Gene ID Conversion: Define g:Profiler API function
# 
# g:Profiler is a web service for functional enrichment analysis and gene ID conversion
# This function converts between different gene identifier systems (Ensembl ID ↔ Gene Symbol)

def gprofiler(gene_query, query_type='ENSEMBL_ID'):
    """
    Use g:Profiler API to convert gene identifiers between different nomenclature systems.
    
    The g:Profiler service (https://biit.cs.ut.ee/gprofiler/) provides comprehensive
    gene ID mapping and functional annotation services.

    Parameters:
    -----------
    gene_query (list): List of gene symbols or IDs to convert
    query_type (str): Type of conversion to perform
                     - 'ENSEMBL_ID': Convert TO Ensembl gene IDs (ENSG format)
                     - 'SYMBOL': Convert TO gene symbols (HGNC format)

    Returns:
    --------
    pandas.DataFrame: DataFrame with conversion results
                     - 'incoming': Original query identifiers
                     - 'converted': Converted identifiers (None if no match found)

    Raises:
    -------
    requests.RequestException: If API request fails (network issues, server down, etc.)
    ValueError: If API returns unexpected response format
    """
    
    # g:Profiler convert API endpoint
    url = 'https://biit.cs.ut.ee/gprofiler/api/convert/convert/'
    
    # API request payload configuration
    payload = {
        'organism': 'hsapiens',              # Human organism
        'target': 'ENSG',                    # Target namespace (Ensembl Gene ID)
        'query': gene_query,                 # List of genes to convert
        'numeric_namespace': 'GENECARDS_ACC', # Numeric identifier handling
    }

    try:
        # Send POST request to g:Profiler API
        response = requests.post(url, json=payload)
        response.raise_for_status()  # Raise exception for HTTP error status codes (4xx, 5xx)
        
    except requests.RequestException as e:
        print(f"API request failed: {e}")
        print("This could be due to network issues or g:Profiler server unavailability")
        return pd.DataFrame(columns=['incoming', 'converted'])  # Return empty DataFrame

    # Parse JSON response
    result = response.json()
    
    # Validate response structure
    if 'result' not in result:
        print("API returned unexpected result format - 'result' key missing")
        return pd.DataFrame(columns=['incoming', 'converted'])

    # Process conversion results
    conversion_dict = {}
    
    for item in result['result']:
        original_query = item.get('incoming', None)    # Original input gene ID
        ensg_id = item.get('converted', None)          # Converted Ensembl ID
        symbol_id = item.get('name', None)             # Gene symbol name
        
        if original_query:
            # Store appropriate conversion based on query type
            if query_type == 'ENSEMBL_ID':
                conversion_dict[original_query] = ensg_id    # Return Ensembl IDs
            elif query_type == 'SYMBOL':
                conversion_dict[original_query] = symbol_id  # Return gene symbols

    # Handle genes that couldn't be converted (set to None)
    for query in gene_query:
        if query not in conversion_dict:
            conversion_dict[query] = None
            
    # Convert to DataFrame for easier handling
    return pd.DataFrame(conversion_dict.items(), columns=['incoming', 'converted'])

In [ ]:
# Gene ID Extraction: Get list of Ensembl gene IDs from expression matrix
# 
# Extract all unique gene identifiers from the expression matrix index
# These are currently in Ensembl format (ENSG...) and need conversion to gene symbols
gene_ensembl_ids = bulkRNA_df.index.tolist()
print(f"Found {len(gene_ensembl_ids)} unique genes in expression matrix")

In [ ]:
# Gene ID Conversion: Convert Ensembl IDs to gene symbols
# 
# Use g:Profiler API to convert Ensembl gene IDs to HGNC gene symbols
# Gene symbols are more interpretable and required for pySCENIC analysis
print("Converting Ensembl IDs to gene symbols using g:Profiler...")
ensembl_ids_to_symbols = gprofiler(gene_ensembl_ids, query_type='SYMBOL')

# Create mapping dictionary for easy lookup during DataFrame index renaming
ensembl_ids_to_symbols_mapping = dict(zip(ensembl_ids_to_symbols['incoming'], ensembl_ids_to_symbols['converted']))

# Report conversion statistics
successful_conversions = sum(1 for v in ensembl_ids_to_symbols_mapping.values() if v is not None)
print(f"Successfully converted {successful_conversions}/{len(gene_ensembl_ids)} genes to symbols")

In [ ]:
# Index Renaming: Replace Ensembl IDs with gene symbols in expression matrix
# 
# Update the DataFrame index to use gene symbols instead of Ensembl IDs
# This makes the data more interpretable and compatible with pySCENIC requirements
print("Renaming gene indices from Ensembl IDs to gene symbols...")
bulkRNA_df.rename(index=ensembl_ids_to_symbols_mapping, inplace=True)

# Remove genes that couldn't be converted (have None as symbol)
original_gene_count = bulkRNA_df.shape[0]
bulkRNA_df = bulkRNA_df.dropna(axis=0)  # Remove rows where index is None
print(f"Kept {bulkRNA_df.shape[0]} genes with valid symbols (removed {original_gene_count - bulkRNA_df.shape[0]} genes)")

In [ ]:
# Data Export: Save expression matrix with gene symbols
# 
# Save the updated expression matrix with gene symbols as row indices
# This serves as the processed input for downstream pySCENIC analysis
output_file_symbols = OUTPUT_PATH + 'renamed_bulkRNA_gene_count_' + TARGET_DISEASE + '_' + NC_STR + 'symbol_' + current_date + '.tsv'
bulkRNA_df.to_csv(output_file_symbols, sep='\t', encoding='utf-8')
print(f"Saved symbol-annotated expression matrix to: {output_file_symbols}")

In [ ]:
# Data Reloading: Load processed expression matrix with gene symbols
# 
# Reload the expression matrix with gene symbols for continued analysis
# This step ensures we're working with the properly formatted data
input_file_symbols = OUTPUT_PATH + 'renamed_bulkRNA_gene_count_' + TARGET_DISEASE + '_' + NC_STR + 'symbol_' + current_date + '.tsv'
bulkRNA_df = pd.read_csv(input_file_symbols, sep='\t', index_col=0)
print(f"Reloaded expression matrix: {bulkRNA_df.shape[0]} genes × {bulkRNA_df.shape[1]} samples")

In [ ]:
# Gene Filtering: Filter genes based on expression variability
# 
# pySCENIC works best with highly variable genes that show meaningful expression differences
# Filter out lowly expressed and non-variable genes to improve analysis quality and speed

print("Performing gene filtering based on expression statistics...")

# Calculate expression statistics for each gene across all samples
mean_expression = bulkRNA_df.mean(axis=1)     # Average expression per gene
std_expression = bulkRNA_df.std(axis=1)       # Standard deviation per gene

# Calculate coefficient of variation (CV = std/mean)
# CV measures relative variability - high CV indicates genes with variable expression
cv = std_expression / mean_expression

# Apply filtering criteria:
# 1. Mean expression > 1: Remove very lowly expressed genes (likely noise)
# 2. Coefficient of variation > 0.5: Keep only genes with substantial variability
selected_genes = bulkRNA_df[(mean_expression > 1) & (cv > 0.5)].index

# Create filtered expression matrix with only selected genes
bulkRNA_df_filtered = bulkRNA_df.loc[selected_genes]

print(f"Gene filtering results:")
print(f"  Original genes: {bulkRNA_df.shape[0]}")
print(f"  After filtering: {bulkRNA_df_filtered.shape[0]} ({bulkRNA_df_filtered.shape[0]/bulkRNA_df.shape[0]*100:.1f}%)")
print(f"  Filtering criteria: mean > 1 AND coefficient of variation > 0.5")

In [ ]:
# Transcription Factor Loading: Load human transcription factor gene list
# 
# Load curated list of human transcription factors for GRN inference
# Only these genes will be considered as potential regulators in the network
print("Loading human transcription factor gene list...")
tf_names = load_tf_names(MM_TFS_FNAME)
print(f"Loaded {len(tf_names)} transcription factor genes")

In [ ]:
# Motif Database Loading: Load cisTarget motif ranking databases
# 
# These databases contain pre-computed motif enrichment rankings for all genes
# Used in step 2 of pySCENIC pipeline for regulon prediction (cisTarget)

print("Loading cisTarget motif ranking databases...")
# Find all feather database files matching the pattern
db_fnames = glob.glob(DATABASES_GLOB)

# Helper function to extract database name from file path
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]

# Create RankingDatabase objects for each database file
# Multiple databases provide comprehensive motif coverage
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]

print(f"Loaded {len(dbs)} motif ranking databases:")
for db in dbs:
    print(f"  - {db.name}")

dbs

In [ ]:
# Data Transposition: Transpose expression matrix for pySCENIC input format
# 
# pySCENIC expects samples as rows and genes as columns
# Original format: genes × samples → Required format: samples × genes
print("Transposing expression matrix for pySCENIC input format...")
bulkRNA_transposed_df = bulkRNA_df.T
print(f"Transposed matrix: {bulkRNA_transposed_df.shape[0]} samples × {bulkRNA_transposed_df.shape[1]} genes")

In [ ]:
# STEP 1: Gene Regulatory Network (GRN) Inference using GRNBoost2
# 
# GRNBoost2 is a scalable algorithm for inferring gene regulatory networks
# from expression data using gradient boosting machines
# It identifies potential transcriptional regulatory relationships

print("Starting GRN inference with GRNBoost2...")
print("This may take several minutes depending on data size...")

# Run GRNBoost2 algorithm
# - Input: transposed expression matrix (samples × genes)
# - tf_names: restrict regulators to known transcription factors
# - verbose: show progress information
adjacencies = grnboost2(bulkRNA_transposed_df, tf_names=tf_names, verbose=True)

print(f"GRN inference completed. Found {len(adjacencies)} potential regulatory interactions.")

In [ ]:
# Export Adjacencies: Save GRN results for future use
# 
# Save the adjacency matrix (TF-target relationships) as a tab-separated file
# This allows reloading the GRN results without re-running the time-intensive inference
adjacencies.to_csv(ADJACENCIES_FNAME, index=False, sep='\t', encoding='utf-8')
print(f"Saved GRN adjacencies to: {ADJACENCIES_FNAME}")

In [ ]:
# Load Adjacencies: Reload previously computed GRN results
# 
# Load the adjacency matrix from file for continued analysis
# This allows resuming analysis without re-running GRN inference
ADJACENCIES_FNAME_TOLOAD = os.path.join(OUTPUT_PATH, 'adjacencies_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.tsv')
adjacencies = pd.read_csv(ADJACENCIES_FNAME_TOLOAD, sep='\t')
print(f"Reloaded {len(adjacencies)} regulatory interactions from {ADJACENCIES_FNAME_TOLOAD}")

In [ ]:
# Module Identification: Create co-expression modules from GRN adjacencies
# 
# Group genes into modules based on their regulatory relationships
# Modules represent sets of genes that are co-regulated by the same transcription factors
print("Creating co-expression modules from GRN adjacencies...")
modules = list(modules_from_adjacencies(adjacencies, bulkRNA_transposed_df))
print(f"Identified {len(modules)} co-expression modules")

In [ ]:
# Save Modules: Export co-expression modules for future use
# 
# Save modules as pickle file to preserve the complex data structure
with open(MODULES_FNAME, 'wb') as f:
    pickle.dump(modules, f)
print(f"Saved {len(modules)} modules to: {MODULES_FNAME}")

In [ ]:
# Load Modules: Reload previously computed co-expression modules
# 
# Load modules from pickle file for continued analysis
with open(MODULES_FNAME, 'rb') as f:
   modules = pickle.load(f)
print(f"Reloaded {len(modules)} co-expression modules from {MODULES_FNAME}")

In [ ]:
# STEP 2: Regulon Prediction using cisTarget motif analysis
# 
# Perform motif enrichment analysis to identify which TFs likely regulate each module
# This step validates predicted TF-target relationships using DNA-binding motif data
print("Performing cisTarget motif enrichment analysis...")
print("This step validates TF-target relationships using DNA-binding motifs...")
df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME)
print("Motif enrichment analysis completed.")

In [ ]:
# Export Motif Results: Save motif enrichment analysis results
# 
# Save the motif enrichment DataFrame as CSV file
# Data contains ';' symbols in some fields, so CSV comma separation is appropriate
df.to_csv(MOTIFS_FNAME, encoding='utf-8')
print(f"Saved motif enrichment results to: {MOTIFS_FNAME}")

In [ ]:
# Regulon Creation: Convert motif enrichment results to final regulons
# 
# Create regulons (TF + target gene sets) from validated motif enrichment results
# Regulons represent the final predicted regulatory relationships
print("Creating final regulons from motif enrichment results...")
regulons = df2regulons(df)
print(f"Created {len(regulons)} validated regulons")

In [ ]:
# Save Regulons: Export final regulons for future analysis
# 
# Save regulons as pickle file to preserve the complex regulon objects
with open(REGULONS_FNAME, 'wb') as f:
    pickle.dump(regulons, f)
print(f"Saved {len(regulons)} regulons to: {REGULONS_FNAME}")

In [ ]:
# Load Regulons: Reload previously computed regulons
# 
# Load final regulons from pickle file for activity analysis
with open(REGULONS_FNAME, 'rb') as f:
   regulons = pickle.load(f)
print(f"Reloaded {len(regulons)} regulons from {REGULONS_FNAME}")

In [ ]:
# STEP 3: Activity Scoring using AUCell
# 
# Calculate Area Under the Curve (AUC) scores for regulon activity in each sample
# AUC scores indicate how active each regulon is in each individual sample
print("Calculating regulon activity scores using AUCell...")
print("This step quantifies regulon activity in each sample...")
auc_mtx = aucell(bulkRNA_transposed_df, regulons, num_workers=1)
print(f"Generated AUC matrix: {auc_mtx.shape[0]} samples × {auc_mtx.shape[1]} regulons")

In [ ]:
# Sample Labeling: Add disease/control labels to AUC matrix sample names
# 
# Rename sample IDs to include disease status for easier interpretation in downstream analysis
# This helps distinguish between disease samples (RP) and normal controls (NC)

print("Adding disease/control labels to sample names...")
samples_renamed = 0

for patient_id in auc_mtx.index:
    # Check if sample belongs to target disease group
    if patient_id[:5] in patient_ids_TARGET.values:
        auc_mtx.rename(index={patient_id: TARGET_DISEASE + '_' + patient_id}, inplace=True)
        samples_renamed += 1
    # Check if sample belongs to normal control group (if included)
    elif INCLUDE_NC and patient_id[:5] in patient_ids_NC.values:
        auc_mtx.rename(index={patient_id: 'NC_' + patient_id}, inplace=True)
        samples_renamed += 1

# Sort sample names in ascending alphabetical order for consistent ordering
auc_mtx = auc_mtx.sort_index()
print(f"Renamed {samples_renamed} samples with disease/control labels")

In [ ]:
# Index Labeling: Set descriptive name for AUC matrix sample axis
# 
# Add a meaningful name to the index (row names) for clearer data interpretation
auc_mtx.index.name = 'Samples'
print("Set AUC matrix index name to 'Samples'")

In [ ]:
# Sample Ordering: Reorder samples for visualization (disease samples first, controls second)
# 
# Sort samples to group disease samples (RP) before control samples (NC)
# Reverse sort puts RP (higher alphabetically) before NC for better visualization
auc_mtx = auc_mtx.reindex(sorted(auc_mtx.index, reverse=True))
print("Reordered samples: disease samples first, control samples second")

In [ ]:
# Visualization: Generate clustered heatmap of regulon activities
# 
# Create a clustermap to visualize regulon activity patterns across samples
# Clustering helps identify sample groups and regulon co-activity patterns
print("Generating clustered heatmap of regulon activities...")
sns.clustermap(auc_mtx, figsize=(30,30))

In [ ]:
# Alternative Visualization: Generate heatmap without row clustering
# 
# Create a clustermap that preserves the sample ordering (no row clustering)
# This maintains the disease vs control grouping for easier comparison
print("Generating heatmap with preserved sample ordering (no row clustering)...")
ax_no_row_cluster = sns.clustermap(auc_mtx, figsize=(30,30), row_cluster=False)

In [ ]:
# Figure Export: Save high-quality plot for publication
# 
# Export the regulon activity heatmap as high-resolution PNG file
# Configure export parameters for publication-quality figures
output_plot_file = OUTPUT_PATH + 'plot/' + 'auc_mtx_clustermap_' + TARGET_DISEASE + '_' + NC_STR + current_date + '.png'
plt.savefig(output_plot_file, dpi=300, bbox_inches='tight')
print(f"Saved high-quality plot to: {output_plot_file}")

In [ ]:
# Display Plot: Show the final regulon activity heatmap
# 
# Display the generated heatmap in the notebook for interactive viewing
plt.show()